# Experiment 8 - Domain-Specific RAG Chatbot for SPIT
Run top to bottom. Set your LLM credentials first (see README).

## Setup

In [ ]:
import os
# os.environ['ANTHROPIC_API_KEY'] = '...'   # or set LLM_PROVIDER / OPENAI_* for another approved LLM
import pandas as pd
import config as C, rag, evaluate
pd.set_option('display.max_colwidth', 120); pd.set_option('display.width', 200)
print(C.EMBED_MODEL, C.CHUNK_TOKENS, C.CHUNK_OVERLAP, C.LLM_PROVIDER, C.LLM_MODEL)

## Task 1 - Knowledge base
Run once (needs internet). Then inspect `knowledge_base/` and `metadata.csv`.

In [ ]:
!python build_kb.py
pd.read_csv(C.META_CSV)

## Tasks 2-3 - Extract, clean, chunk

In [ ]:
docs = rag.load_corpus()
chunks = rag.build_chunks(docs)
print(len(docs), 'documents ->', len(chunks), 'chunks')
pd.DataFrame(chunks)[['chunk_id','source','page_or_section','text']].head(10)

## Tasks 4-5 - Embeddings + FAISS index

In [ ]:
embedder = rag.Embedder()
store, tag = rag.build_store(embedder)   # also saves index/<tag>/ (faiss index, embeddings, chunks.jsonl/csv)
store.index.ntotal, tag

## Task 6 - Retrieval (retrieved text + source + score)  -  screenshot this for >= 5 queries

In [ ]:
for q in ['Where is SPIT located?', 'Which university is SPIT affiliated to?']:
    print('='*100); print('Q:', q)
    rag.show_results(rag.retrieve(store, embedder, q, k=3))

## Tasks 7-8 - Prompt + grounded answer

In [ ]:
out = rag.rag_answer('Where is SPIT located?', store, embedder, k=3)
print(out['prompt'][:1500], '\n...\n')
print('ANSWER:', out['answer'])
print(f"retrieval {out['t_retrieve']:.2f}s, generation {out['t_generate']:.2f}s")

### Unanswerable question (must refuse, not fabricate)

In [ ]:
out = rag.rag_answer('What is the Wi-Fi password of the SPIT campus network?', store, embedder, k=3)
rag.show_results(out['results'], 200); print('ANSWER:', out['answer'])

## Evaluation set

In [ ]:
qa = evaluate.load_qa('qa_set.csv')
qa[['qid','type','question','answerable']]

## Experiment A - Top-k (k = 1, 3, 5, 10)

In [ ]:
tableA = evaluate.exp_a(qa, tag, judge=True)
tableA

Observation sheet (k=3) - fill 'Notes' and verify by hand

In [ ]:
evaluate.observation_sheet(pd.read_csv(C.RESULTS_DIR/'expA_answers_k3.csv')).head(30)

## Experiment B - Retrieval failure (edit `remove` to a source holding some answers)

In [ ]:
evaluate.exp_b(qa, remove='about/', tag=tag, judge=True)

## Experiment C - Irrelevant / conflicting context

In [ ]:
evaluate.exp_c(qa, tag, judge=True)

## Experiment D - RAG vs ICL

In [ ]:
evaluate.exp_d(qa, tag, judge=True)

## Advanced challenge - chunking, embeddings, MMR, hybrid, threshold

In [ ]:
evaluate.exp_adv(qa)

## Analysis
Answer the 9 analysis questions from the manual here, using your tables, plus at least two retrieval-failure cases and the final conclusion.